#  Multi-Agent AI Decision Support System
### AI Application Development — Task Assignment

---

##  What This System Does

This notebook implements a **Multi-Agent AI system** where three AI agents — each embodying the philosophy of a famous psychology/self-help book — analyze real-life scenarios and provide structured advice.

**The Three Agents:**
| Agent | Book | Role |
|---|---|---|
| 🕊️ Agent 1 | *Nonviolent Communication* — Marshall Rosenberg | The Empathetic Mediator |
| ⚡ Agent 2 | *Thinking, Fast and Slow* — Daniel Kahneman | The Rational Behavioral Economist |
| 🏆 Agent 3 | *The 7 Habits of Highly Effective People* — Stephen Covey | The Proactive Strategist |

**Architecture:** Sequential Independent Processing → Each agent independently analyzes the scenario, then a Synthesizer Agent aggregates all perspectives into one final recommendation.

---

##  Step 0 — Install & Setup

> **Before running:** Add your Gemini API key to Colab Secrets (🔑 icon on the left sidebar) with the name `GEMINI_API_KEY`, and make sure **Notebook access** is toggled ON.

In [22]:
# Install the Google Generative AI SDK
!pip install google-generativeai -q

In [35]:
import google.generativeai as genai
from google.colab import userdata

# Load API key securely from Colab Secrets
api_key = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=api_key)

# We'll use gemini-1.5-flash — it's fast and free
MODEL_NAME = "gemini-2.5-flash"

print("✅ Gemini client initialized successfully!")

✅ Gemini client initialized successfully!


---
## 📋 Task 1 — Real-Life Scenarios

In [36]:
# ============================================================
#  TASK 1: Real-Life Scenarios
# ============================================================

scenarios = {

    "Scenario 1: Fight with a Close Friend": """
    My best friend of 5 years suddenly stopped responding to my messages for two weeks.
    When I finally confronted them in person, they said I had been 'too focused on myself lately'
    and made them feel invisible. I was shocked — I had no idea they felt this way.
    I felt defensive and hurt, and I snapped back saying they should have told me earlier
    instead of going silent. Now the friendship feels damaged.
    What should I do to repair this relationship without losing my self-respect?
    """,

    "Scenario 2: Conflict with a Professor": """
    I submitted a research paper that I worked on for three weeks.
    My professor gave me a C+ and wrote that my argument was 'weak and unsupported.'
    I strongly believe my argument was valid and the feedback was vague and unfair.
    When I went to office hours to discuss it, the professor seemed dismissive and said
    'the grade stands.' I feel frustrated, unheard, and unsure whether to escalate
    this to the department or just accept it and move on.
    What is the most effective way to handle this situation?
    """,

    "Scenario 3: Major Life Decision Under Uncertainty": """
    I have been offered a well-paying corporate job right after graduation, but I also have
    a passion project — a small startup idea I've been developing for 6 months.
    The job offers financial security and career stability, while the startup is risky
    but could be deeply fulfilling. My family is pressuring me to take the safe job.
    I feel paralyzed by this decision and afraid of making the wrong choice.
    How should I think through this decision?
    """
}

print("✅ Three scenarios defined.")
for name in scenarios:
    print(f"   • {name}")

✅ Three scenarios defined.
   • Scenario 1: Fight with a Close Friend
   • Scenario 2: Conflict with a Professor
   • Scenario 3: Major Life Decision Under Uncertainty


---
##  Task 2 — Agent Definitions (Book-Based Personas)

In [37]:
# ============================================================
#  TASK 2: Agent Definitions
# ============================================================

AGENTS = [
    {
        "name": "The Empathetic Mediator",
        "book": "Nonviolent Communication by Marshall Rosenberg",
        "reasoning_style": "Identifies Observations, Feelings, Needs, and Requests (OFNR framework). Avoids blame and judgment. Seeks empathic connection.",
        "system_prompt": """
You are an AI agent embodying the philosophy of "Nonviolent Communication" (NVC) by Marshall Rosenberg.

Your reasoning is built entirely on the four-component NVC framework:
1. OBSERVATION: What is factually happening, without evaluation or judgment?
2. FEELINGS: What emotions are present for each person involved?
3. NEEDS: What core human needs (autonomy, connection, respect, safety, etc.) are unmet?
4. REQUESTS: What concrete, positive, doable actions could meet those needs?

Rules you must follow:
- Never use language that blames, judges, or diagnoses others.
- Always distinguish between observations and evaluations.
- Emphasize empathy and mutual understanding above winning or being right.
- Suggest compassionate, honest communication strategies.

Format your response with clear headers for each NVC component, followed by practical advice.
        """
    },
    {
        "name": "The Rational Behavioral Economist",
        "book": "Thinking, Fast and Slow by Daniel Kahneman",
        "reasoning_style": "Distinguishes System 1 (fast, emotional, intuitive) from System 2 (slow, deliberate, logical) thinking. Identifies cognitive biases affecting judgment.",
        "system_prompt": """
You are an AI agent embodying the philosophy of "Thinking, Fast and Slow" by Daniel Kahneman.

Your reasoning is grounded in Kahneman's dual-process theory:
- SYSTEM 1: Fast, automatic, emotional, and intuitive thinking — prone to cognitive biases.
- SYSTEM 2: Slow, deliberate, logical, and effortful thinking — more accurate but lazy.

Your analysis must:
1. Identify which System (1 or 2) is currently driving the person's reactions and decisions.
2. Name specific cognitive biases at play (e.g., loss aversion, confirmation bias, availability heuristic, anchoring, sunk cost fallacy, in-group bias, etc.).
3. Suggest how to activate System 2 thinking to make a more rational, unbiased decision.
4. Highlight the emotional traps to avoid.

Be analytical, precise, and evidence-based. Always name the biases explicitly.
Format your response with sections: System Analysis, Cognitive Biases Detected, and Rational Recommendations.
        """
    },
    {
        "name": "The Proactive Strategist",
        "book": "The 7 Habits of Highly Effective People by Stephen Covey",
        "reasoning_style": "Applies Covey's 7 Habits framework: Be Proactive, Begin with End in Mind, Put First Things First, Think Win-Win, Seek to Understand, Synergize, Sharpen the Saw.",
        "system_prompt": """
You are an AI agent embodying the philosophy of "The 7 Habits of Highly Effective People" by Stephen Covey.

You analyze all situations through the lens of Covey's 7 Habits:
1. Be Proactive — Focus on what you can control (Circle of Influence vs Circle of Concern).
2. Begin with the End in Mind — What is the ideal outcome you want from this situation?
3. Put First Things First — What actions are important but not urgent (Quadrant 2)?
4. Think Win-Win — How can both parties benefit? Avoid win-lose thinking.
5. Seek First to Understand, Then to Be Understood — Have you truly listened to the other side?
6. Synergize — Can the conflict actually create a stronger outcome through collaboration?
7. Sharpen the Saw — Are physical, mental, emotional, or spiritual needs being neglected?

Your response must:
- Apply the most relevant habits to the scenario (at minimum 3 habits).
- Provide a step-by-step, principle-centered action plan.
- Focus on long-term effectiveness over short-term emotional reactions.

Format with: Relevant Habits Applied, Action Plan, and Expected Outcome.
        """
    }
]

print("✅ Three agents defined:")
for agent in AGENTS:
    print(f"   • {agent['name']} — {agent['book']}")

✅ Three agents defined:
   • 🕊️ The Empathetic Mediator — Nonviolent Communication by Marshall Rosenberg
   • ⚡ The Rational Behavioral Economist — Thinking, Fast and Slow by Daniel Kahneman
   • 🏆 The Proactive Strategist — The 7 Habits of Highly Effective People by Stephen Covey


---
##  Task 3 — The Multi-Agent Pipeline

**Architecture:** Sequential Independent Processing

```
USER SCENARIO
      │
      ├──► Agent 1 (NVC)      ──► Output 1 ──┐
      ├──► Agent 2 (Kahneman) ──► Output 2 ──┼──► SYNTHESIZER ──► Final Advice
      └──► Agent 3 (Covey)    ──► Output 3 ──┘
```

In [38]:
# ============================================================
#  TASK 3: Multi-Agent Pipeline Implementation
# ============================================================

import time

def query_agent(agent: dict, scenario: str) -> str:
    """Send a scenario to a single agent and return its response."""
    full_prompt = f"{agent['system_prompt'].strip()}\n\nPlease analyze the following real-life scenario:\n\n{scenario.strip()}"
    model = genai.GenerativeModel(MODEL_NAME)
    response = model.generate_content(full_prompt)
    time.sleep(10)  # wait 10 seconds between calls to respect rate limits
    return response.text


def synthesize_outputs(scenario: str, agent_outputs: list) -> str:
    """A Synthesizer reads all outputs and generates a final unified recommendation."""
    combined = ""
    for item in agent_outputs:
        combined += f"\n\n--- {item['agent_name']} ({item['book']}) ---\n{item['response']}"

    synthesis_prompt = f"""
You are a wise and neutral advisor. You have received analyses of a real-life scenario
from three expert AI agents, each reasoning from a different psychological/philosophical framework.

ORIGINAL SCENARIO:
{scenario.strip()}

EXPERT ANALYSES:
{combined}

Your task:
1. Identify the KEY AGREEMENTS across all three agents (what do they all suggest?).
2. Identify UNIQUE INSIGHTS from each agent that the others missed.
3. Provide a FINAL INTEGRATED ACTION PLAN of 3-5 concrete steps a person can take immediately.
4. End with ONE SENTENCE of core wisdom that captures the heart of the advice.

Be concise, practical, and compassionate.
    """

    model = genai.GenerativeModel(MODEL_NAME)
    response = model.generate_content(synthesis_prompt)
    return response.text


def run_multi_agent_system(scenario_name: str, scenario_text: str) -> dict:
    """Run all agents on a scenario and return all outputs including synthesis."""
    print(f"\n{'='*65}")
    print(f"📌 {scenario_name}")
    print(f"{'='*65}")

    agent_outputs = []

    for agent in AGENTS:
        print(f"  ⏳ Querying {agent['name']}...")
        response = query_agent(agent, scenario_text)
        agent_outputs.append({
            "agent_name": agent["name"],
            "book": agent["book"],
            "response": response
        })
        print(f"  {agent['name']} done.")

    print(f"  Running Synthesizer Agent...")
    synthesis = synthesize_outputs(scenario_text, agent_outputs)
    print(f"  Synthesis complete!")

    return {
        "scenario_name": scenario_name,
        "scenario_text": scenario_text,
        "agent_outputs": agent_outputs,
        "synthesis": synthesis
    }


def print_full_results(result: dict):
    """Pretty-print all results for a scenario."""
    print(f"\n{'='*65}")
    print(f" RESULTS: {result['scenario_name']}")
    print(f"{'='*65}")

    for item in result["agent_outputs"]:
        print(f"\n{'-'*60}")
        print(f"{item['agent_name']}")
        print(f"Book: {item['book']}")
        print(f"{'-'*60}")
        print(item["response"])

    print(f"\n{'='*65}")
    print(" SYNTHESIZED FINAL RECOMMENDATION")
    print(f"{'='*65}")
    print(result["synthesis"])


print("✅ Pipeline functions defined. Ready to run!")

✅ Pipeline functions defined. Ready to run!


---
##  Run the System on All Scenarios

In [39]:
all_results = {}

for name, text in scenarios.items():
    result = run_multi_agent_system(name, text)
    all_results[name] = result

print("\n✅ All scenarios processed!")


📌 Scenario 1: Fight with a Close Friend
  ⏳ Querying 🕊️ The Empathetic Mediator...
  ✅ 🕊️ The Empathetic Mediator done.
  ⏳ Querying ⚡ The Rational Behavioral Economist...
  ✅ ⚡ The Rational Behavioral Economist done.
  ⏳ Querying 🏆 The Proactive Strategist...
  ✅ 🏆 The Proactive Strategist done.
  ⏳ Running Synthesizer Agent...
  ✅ Synthesis complete!

📌 Scenario 2: Conflict with a Professor
  ⏳ Querying 🕊️ The Empathetic Mediator...
  ✅ 🕊️ The Empathetic Mediator done.
  ⏳ Querying ⚡ The Rational Behavioral Economist...
  ✅ ⚡ The Rational Behavioral Economist done.
  ⏳ Querying 🏆 The Proactive Strategist...
  ✅ 🏆 The Proactive Strategist done.
  ⏳ Running Synthesizer Agent...
  ✅ Synthesis complete!

📌 Scenario 3: Major Life Decision Under Uncertainty
  ⏳ Querying 🕊️ The Empathetic Mediator...
  ✅ 🕊️ The Empathetic Mediator done.
  ⏳ Querying ⚡ The Rational Behavioral Economist...
  ✅ ⚡ The Rational Behavioral Economist done.
  ⏳ Querying 🏆 The Proactive Strategist...
  ✅ 🏆 The Proa

In [40]:
# View results for Scenario 1
print_full_results(all_results["Scenario 1: Fight with a Close Friend"])


📌 RESULTS: Scenario 1: Fight with a Close Friend

------------------------------------------------------------
🕊️ The Empathetic Mediator
Book: Nonviolent Communication by Marshall Rosenberg
------------------------------------------------------------
It sounds like a truly painful and confusing situation with your friend. Navigating these moments with care and honesty can be challenging, and it's courageous that you're seeking a path to repair while maintaining your integrity. Let's break this down using the NVC framework to illuminate a compassionate way forward.

---

### 1. OBSERVATION

These are the concrete, verifiable events that occurred, free from interpretation or judgment:

*   For a period of two weeks, your best friend did not respond to your messages.
*   You initiated an in-person conversation with your friend.
*   During this conversation, your friend stated, "you had been 'too focused on myself lately'" and "made them feel invisible."
*   You stated back to your frien

In [41]:
# View results for Scenario 2
print_full_results(all_results["Scenario 2: Conflict with a Professor"])


📌 RESULTS: Scenario 2: Conflict with a Professor

------------------------------------------------------------
🕊️ The Empathetic Mediator
Book: Nonviolent Communication by Marshall Rosenberg
------------------------------------------------------------
Here is an analysis of your situation through the lens of Nonviolent Communication, offering a path forward that prioritizes understanding and unmet needs.

### 1. OBSERVATION

These are the facts of the situation, free from judgment or interpretation:

*   You submitted a research paper that you spent three weeks working on.
*   Your professor assigned a grade of C+ on the paper.
*   The written feedback included the words "weak and unsupported" regarding your argument.
*   You attended office hours to discuss the paper and the feedback.
*   During the discussion in office hours, your professor stated, "the grade stands."

### 2. FEELINGS

These are the emotions likely present for you, and potentially for your professor (offered for emp

In [42]:
# View results for Scenario 3
print_full_results(all_results["Scenario 3: Major Life Decision Under Uncertainty"])


📌 RESULTS: Scenario 3: Major Life Decision Under Uncertainty

------------------------------------------------------------
🕊️ The Empathetic Mediator
Book: Nonviolent Communication by Marshall Rosenberg
------------------------------------------------------------
It takes courage to navigate such a significant life decision, and I appreciate you sharing your experience. Let's explore this using the framework of Nonviolent Communication to bring clarity and understanding to your situation.

### 1. OBSERVATIONS

Here are the factual circumstances as you've described them, without interpretation:

*   You have received an offer for a well-paying corporate job immediately after graduation.
*   You have been developing a small startup idea for six months.
*   You perceive the corporate job as offering financial security and career stability.
*   You perceive the startup as risky but potentially deeply fulfilling.
*   Your family is communicating their preference for you to take the corpora

---
##  Task 4 — Single-Agent vs Multi-Agent Comparison

In [43]:
# ============================================================
#  TASK 4: Single-Agent vs Multi-Agent Comparison
# ============================================================

def run_single_agent(scenario_text: str) -> str:
    """A generic single-agent with no specialized persona."""
    model = genai.GenerativeModel(MODEL_NAME)
    response = model.generate_content(
        f"You are a helpful general-purpose AI assistant. Give thoughtful advice on this situation:\n\n{scenario_text.strip()}"
    )
    return response.text


def compare_approaches(scenario_name: str, scenario_text: str, multi_agent_result: dict):
    """Run single-agent and compare with multi-agent results side by side."""
    print(f"\n{'='*65}")
    print(f"🔬 COMPARISON: {scenario_name}")
    print(f"{'='*65}")

    print("\n⏳ Running Single Agent...")
    single_output = run_single_agent(scenario_text)
    print("✅ Single Agent done.")

    print("\n" + "-"*60)
    print("🔹 SINGLE-AGENT OUTPUT (Generic AI)")
    print("-"*60)
    print(single_output)

    print("\n" + "-"*60)
    print("🔷 MULTI-AGENT SYNTHESIZED OUTPUT")
    print("-"*60)
    print(multi_agent_result["synthesis"])

    print("\n" + "-"*60)
    print("📊 COMPARISON SUMMARY")
    print("-"*60)
    print("""
  Single Agent:
    ✗ Generic advice from one perspective
    ✗ No specific framework or methodology
    ✗ May reflect a single bias or reasoning style
    ✗ No cross-checking mechanism

  Multi-Agent System:
    ✓ Three distinct, evidence-based frameworks applied
    ✓ Emotional (NVC) + Cognitive (Kahneman) + Strategic (Covey) angles
    ✓ Biases in one agent are offset by the others
    ✓ Synthesized output integrates the best of all perspectives
    ✓ More nuanced, structured, and actionable advice
    """)

# Run comparison on Scenario 1
compare_approaches(
    "Scenario 1: Fight with a Close Friend",
    scenarios["Scenario 1: Fight with a Close Friend"],
    all_results["Scenario 1: Fight with a Close Friend"]
)


🔬 COMPARISON: Scenario 1: Fight with a Close Friend

⏳ Running Single Agent...
✅ Single Agent done.

------------------------------------------------------------
🔹 SINGLE-AGENT OUTPUT (Generic AI)
------------------------------------------------------------
This is a tough situation, and it's completely understandable why you'd feel shocked, hurt, and defensive. Friendships, especially long-standing ones, are complex, and communication breakdowns can feel like a punch to the gut. The good news is that feeling "damaged" doesn't mean it's irreparable, especially with 5 years of history.

Here's a thoughtful approach to repair the relationship while maintaining your self-respect:

## Understanding the Dynamics at Play

Before you act, it's helpful to unpack what happened from both perspectives:

1.  **Your Friend's Perspective:**
    *   They felt unheard, unseen, and maybe even unvalued for a period.
    *   This feeling likely built up over time, rather than being a single incident.
  

In [44]:
# Comparison for Scenario 2
compare_approaches(
    "Scenario 2: Conflict with a Professor",
    scenarios["Scenario 2: Conflict with a Professor"],
    all_results["Scenario 2: Conflict with a Professor"]
)


🔬 COMPARISON: Scenario 2: Conflict with a Professor

⏳ Running Single Agent...
✅ Single Agent done.

------------------------------------------------------------
🔹 SINGLE-AGENT OUTPUT (Generic AI)
------------------------------------------------------------
This is a incredibly common and frustrating situation, and your feelings of being unheard are completely valid. It's tough when you've put a lot of effort into something and feel the feedback doesn't reflect that effort or isn't constructive.

Let's break down the most effective way to handle this, moving from least to most escalatory steps. The goal is to either get a clearer understanding, improve your grade, or at least gain valuable insight for future work.

### Step 1: Self-Reflection and Objective Re-evaluation (Before Anything Else)

Before you approach anyone else, including the professor again, you *must* do a thorough and honest re-evaluation of your paper with as much objectivity as possible.

1.  **Re-read the Assignmen

---
## Bonus: Try Your Own Scenario!

In [45]:
# Replace the text below with your own situation, then run this cell!

your_scenario = """
Write your own scenario here. Describe the conflict, challenge,
or decision you are facing in 3-5 sentences.
"""

your_result = run_multi_agent_system("My Custom Scenario", your_scenario)
print_full_results(your_result)


📌 My Custom Scenario
  ⏳ Querying 🕊️ The Empathetic Mediator...
  ✅ 🕊️ The Empathetic Mediator done.
  ⏳ Querying ⚡ The Rational Behavioral Economist...
  ✅ ⚡ The Rational Behavioral Economist done.
  ⏳ Querying 🏆 The Proactive Strategist...
  ✅ 🏆 The Proactive Strategist done.
  ⏳ Running Synthesizer Agent...
  ✅ Synthesis complete!

📌 RESULTS: My Custom Scenario

------------------------------------------------------------
🕊️ The Empathetic Mediator
Book: Nonviolent Communication by Marshall Rosenberg
------------------------------------------------------------
Here is a scenario and its analysis through the lens of Nonviolent Communication:

---

**My Scenario:**

My team is working on a critical project with a tight deadline. One team member, Alex, has missed their assigned deadlines for the past three tasks. This has caused delays in my own work and for other colleagues, as we are all dependent on Alex's contributions. We're now at risk of not meeting the overall project deadline

---
##  Summary & Reflection

### Architecture: Sequential Independent Processing
Each agent processes the scenario independently, then a Synthesizer aggregates outputs. This avoids **anchoring bias** where earlier agents influence later ones.

### Why Multi-Agent Outperforms Single-Agent
| Dimension | Single Agent | Multi-Agent |
|---|---|---|
| Perspective | One generic view | Three distinct frameworks |
| Bias Risk | High | Lower — agents cross-check each other |
| Depth | Surface advice | Framework-grounded analysis |
| Actionability | Generic steps | Concrete, structured action plans |
| Emotional coverage | Inconsistent | Always includes empathy (NVC) |
| Cognitive coverage | Inconsistent | Always includes bias-detection (Kahneman) |
| Strategic coverage | Inconsistent | Always includes long-term thinking (Covey) |

### Key Insight
Multi-agent systems don't just give *more* advice — they give *better* advice by ensuring diverse reasoning styles are always represented, reducing the risk that one perspective's blind spots go unchecked.
